# Lottery Prediction Program: Dev Notes

uses genetic algorithms to 'evolve' the answer

## Powerball Rules

Select five numbers from 1 to 69 for the white balls; then select one number from 1 to 26 for the red Powerball. Choose your numbers on a play slip or let the lottery terminal randomly pick your numbers. The Powerball jackpot grows until it is won.

## Powerball Prizes

- 1 number plus the Powerball – still $4
- 2 numbers plus the Powerball - $7
- 3 numbers - $7 again
- 3 numbers plus the Powerball - $100
- 4 numbers - $100
- 4 numbers plus the Powerball - $50,000
- 5 numbers - $1 million


Powerball jackpot – The whole lot

### Multiplier

For an additional $1 per play, the Power Play feature can multiply non-jackpot prizes by 2, 3, 4, 5 or 10 times! The multiplier number is randomly selected just before each drawing. The 10X multiplier is only in play when the advertised jackpot annuity is $150 million or less.

## todo

- [X] create sqlite database (stores lottery results)
- [X] download lottery results
- [ ] save / update database
- [x] logging
- [X] genetic algortith
  - [X] Initial population
  - [X] Fitness function
  - [X] Selection
  - [X] Crossover
  - [X] Mutation

## References

[Introduction to Genetic Algorithms](https://towardsdatascience.com/introduction-to-genetic-algorithms-including-example-code-e396e98d8bf3)

In [ ]:
#!/usr/bin/env python3

# Import Libraries
import pandas as pd
import os
import sys
import logging
import json
import random
import sqlite3
# from sqlalchemy import create_engine
from datetime import date
from datetime import datetime


In [ ]:
# Define some global variables
lotdbloc=os.environ.get('LOTTERY_DB', './') # location of sqlite repository
userhomedir=os.path.expanduser( '~' )
# lottery url
powerballurl='https://data.ny.gov/api/views/d6yy-54nr/rows.csv?accessType=DOWNLOAD'
# Number of generations to iterate througn
generations=1000
# sqllite database location
dbloc='./'
dbname='lottery.sqlite'



## Define ball class

In [ ]:
# Define ball class
class Ball:

    # Class Variables

    mutlevel = 5
    crossovermax = 3  # reps max boundery for binary crossover can be 1-7
    powerballmax = 26  # largest powerball number
    ballmax = 69  # largest normal ball value

    def __init__(self, name, balldata) -> None:
        self.name = name
        self.firstfit = 0  # most fit number
        self.secondfit = 0  # second most fit number
        self.newnumber = 0  # The evolved number
        self.data = self.__fitness__(balldata)
        self.ballpos = self.__ballpos__(name)
        # self.selection(self.data,minvalue)

    def selection(self, minvalue):
        ######### Seelection ##############
        # creates a pool of the fittest (highest ranking numbers. the fit pull is of random size up to half the size of the whole range
        # The powerball doesn't have a min value so we set it to zero if the ball is the powerball and we set the maxvalue to the powerball maxvalue (26) which is different from the normal ballmax value (69)
        if self.name == 'powerball':
            maxvalue = self.powerballmax
            minvalue = 1
        else:
            # if it's not the powerball set the mazvalue to ballmax (69)
            maxvalue = self.ballmax
        """
        The fitpool is the pool of numbers from which the first and second fit numbers will be chosen.
        It is comprised first from the list of numnbers in ball.data that range from minvalue to maxvalue. Except fo the powerball minvalue is initially 1 and then set to the value of the previous ball drawn
        if the size of the fitpool is 5 or less 
        """
        fitpoolsize = int(
            self.data.iloc[minvalue:maxvalue].size)  # get the size of the fitpool. The fit pool is made up of the numbers between the min and max value
        if 1 < fitpoolsize <= 5:  # if the fitpool is between 2 and five numbers then the fitpool is all the numbers between minvalue and maxvalue
            fitpool = self.data.iloc[minvalue:maxvalue]
            # fitpool = self.data.iloc[minvalue]
        elif fitpoolsize < 2:  # the fit pool has to be atleast 2 so if the fitpoolsize is less than 2 then through an error to the logger an exit
            errmsg = (
                f"Need at least 2 values for the fit pool.\nballname= {self.name} min ={minvalue} max= {maxvalue}\n Fitpool= {self.data.iloc[minvalue:maxvalue]}")
            logger.warning(errmsg)
            exit()
        else:  # Else in all other circumstances make the fitpool a random size made up of the top halp of all possible values between minvalue and mazvalue
            fitpool = self.data.iloc[minvalue:maxvalue].nlargest(
                random.randrange(2, int(fitpoolsize/2)+1))
        # pick the list of numbers that match a randomly chosen range from the fit pool. In most cases this should be 1 but somethines two or more numbers can have the same rank
        z = self.data.index[self.data ==
                            fitpool[random.randrange(fitpool.size)]].tolist()
        # TO pick the first fit number if this is the first ball and newnumber hasn't been assigned, or new number is below or above the min and maxvalue choose a value for first fit from the fitpool otherwise assign the newly generated number to first fit so it can continue to evolve over generations.
        # if z contains more than 1 number pick one at random and assign it to first fit
        if self.newnumber == 0 or \
                self.newnumber < minvalue or \
                (self.name != 'Ball5' and self.newnumber <= int(maxvalue-(5-self.ballpos))) or \
                self.newnumber > maxvalue:
            self.firstfit = int(z[random.randrange(len(z))])
        else:
            self.firstfit = int(self.newnumber)
        while self.firstfit in z:  # to make sure second fit cannot equal first fit keep changing the fitpool until is doesn't have firstfit in it then pick a random value from the fit pool for second fit
            z = self.data.index[self.data ==
                                fitpool[random.randrange(fitpool.size)]].tolist()
        self.secondfit = int(z[random.randrange(len(z))])
        ###################################

    def mutation(self, val, mutlevel):
        """ 
        This method mutates a binary by flipping it's value based on mutlevel which is a percentage value. if mutlevel is 100 then mutation will always occur, if it's 50
        then it will occur 50% of the time
        """
        if random.randrange(0, 100) < mutlevel:
            if val[0] == 1:
                x = 0
            else:
                x = 1
        else:
            x = val[0]
        return str(x)

    def crossover(self, minvalue):
        """ 
        Mates the fittest numbers to produce a new, fitter number
        """
        if self.name == 'powerball':
            minvalue = 1
        self.selection(minvalue)  # selects the fitpool for the ball
        a = [x for x in str(bin(int(self.firstfit)))[2:].rjust(
            7, '0')]  # turns firstfit into a binary representation
        # turns secondfit into a binary representation
        b = [x for x in str(bin(int(self.secondfit)))[2:].rjust(7, '0')]

        # represents the buts elegable for crossover - this is a random number upto crossovermaz
        boundlist = [x for x in range(1, self.crossovermax+1)]
        # assigned a weighted to the corssover bounderies so that the smallest bit has a x 20 change of being changed over the largest bit
        weightlist = [20, 15, 10, 7, 5, 4, 1]
        newvalue = self.firstfit
        # ensures that crossover doesn't inadvertantly breed a smaller number than minvalue or greater values that ballmax or powerballmax
        while (newvalue <= minvalue) or (self.name in ['Ball1', 'Ball2', 'Ball3', 'Ball4', 'Ball5'] and newvalue > self.ballmax-5-self.ballpos) or (self.name == 'powerball' and newvalue > self.powerballmax):
            # Performs the crossover
            boundary = random.choices(
                boundlist, weights=weightlist[:self.crossovermax], k=1)[0]
            for i in range(boundary):
                abit = a[len(a)-1-i]
                bbit = b[len(b)-1-i]
                a[len(a)-1-i] = self.mutation(bbit, self.mutlevel)
                b[len(b)-1-i] = self.mutation(abit, self.mutlevel)
            # turns the new value back to a decimal int from a string
            newvalue = int("".join(a), 2)
            # check if new crossover has bred a nunber larger than the max values - if so return 0
        # if self.name in ['Ball1','Ball2','Ball3','Ball4','Ball5'] and newvalue>self.ballmax-5-self.ballpos:
        #     return 0
        # elif self.name == 'powerball' and newvalue>self.powerballmax:
        #     return 0
        # else:
        return newvalue

    def __fitness__(self, series):
        """ 
        Accepts a single column dataframe as input `series`
        
        Works out the frequency of each number in the series and passes this back as a dictionany in the format:
        {'number': frequency as percentage of all draws}
        
        it then selects the first and second fittest (with a little randomness) and writes those back to self.firstfit and self.secondfit
        
        this function is called by the __init__ method to populate self.data
        """
        ######### Population ##############
        # puts each ball value and it's frequency as a percentage into the class `data` which is of panda type 'series'
        # returns frequency as a decimal percentage
        x = series.value_counts(normalize=True).sort_index()
        if self.name == 'powerball':
            maxvalue = self.powerballmax
        else:
            maxvalue = self.ballmax
        y = x.reindex([f"{i:02}" for i in range(1, maxvalue+1)], fill_value=0)

        return y  # returns the population

    def __ballpos__(self, name):
        """ 
        gives the ball a numeric value based on it's position
        """
        if name == 'ball1':
            return 1
        elif name == 'ball2':
            return 2
        elif name == 'ball3':
            return 3
        elif name == 'ball4':
            return 4
        elif name == 'ball5':
            return 5
        elif name == 'powerball':
            return 6
        else:
            return 'err'


## logging

Uses the standard python logger.

For is machines checks that ~/Libaray/Logs directory exists and puts `lottery.log` there. If not saves it to the current directory program is executing from.

Logger is also configured to write last log entry to `stdout`

### Logging Levels

```py
logger.debug("Harmless debug Message")
logger.info("Just an information")
logger.warning("Its a Warning")
logger.error("Did you try to divide by zero")
logger.critical("Internet is down")
```

In [ ]:
# Configures logging
# set logfile path
if os.path.isdir(f"{userhomedir}/Library/Logs"):
  logfilepath=f"{userhomedir}/Library/Logs"
else:
  logfilepath=os.getcwd()
logfile=(f"{logfilepath}/lottery.log")
# inits basic logging format
logging.basicConfig(filename=logfile,
                    format='%(asctime)s %(message)s',
                    filemode='a+')
logger = logging.getLogger() # creates logging object
logger.setLevel(logging.DEBUG) # sets logging threshhold
# defines an error handler that will stream the error message to stdout instead of stderr so whenever I write to the logger it will also write to the screen
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.DEBUG)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

## Check Config 

Make sure that any dependencies are in place, like the database and it's location

In [ ]:
def config_check():
  # Check for config errors
  if lotdbloc==None:
    errmsg="No database location is defined for $LOTTERY_DB"
    logger.warning(errmsg)
    return False
  elif generations==None:
    errmsg="The Generations variable is not set"
    logger.warning(errmsg)
  # elif mutlevel==None or (mutlevel > 0 and mutlevel <=100):
  #   errmsg="mutlevel must be a value between 1 and 100"
  #   logger.warning(errmsg)
  # elif crossovermax==None or 0<crossovermax<=7:
  #   errmsg="crossovermax must be a value between 1 and 7"
  #   logger.warning(errmsg)
  else:
    return True

## Get powerball data

URl = `https://data.ny.gov/api/views/d6yy-54nr/rows.csv?accessType=DOWNLOAD`

The number data comes in as a single space delimited column called 'Winning Data' so this has to be parsed into individual columns, in code

In [ ]:
def get_numbers(url):
  """ 
  This method downloads the powerball history from the website and transforms and returns it it as a Pandas dataframe
  """
  # reads the raw csv into a dataframe
  rawhist=pd.read_csv(url)
  # creates a new dataframe where the single column 'winning data' is parsed into 6 columns, 1 for each ball
  hist=rawhist[['Draw Date','Multiplier']].join(rawhist['Winning Numbers'].str.split(expand=True))
  # rename the columns so that each ball has a meaningfull column name
  hist.columns=['Date','Multiplier','Ball1','Ball2','Ball3','Ball4','Ball5','powerball']
  return hist

In [ ]:
def gagetvalues(balls, gens):
  """ 
  This method uses genetic algoritms to evolve the value of the ball. it is passed:
  balls - The list of ball object objects (lball)
  gens - the number of generations to iterate over
  """
  newballs = balls  # new balls starts as balls but will evolve into the list of new balls
  for i in range(gens):  # modify newballs gens times - ths evolves over x generations
    results = []
    # minimum value that the newnumber value must be greater than (as balls rise in value sequentially)
    minnum = 0
    for j in newballs:  # loop through all the balls in the list and generate new numbers
      # j.newnumber = j.crossover(minnum)
      while j.newnumber ==0:
        j.newnumber = j.crossover(minnum)

      j.secondfit = j.firstfit # make the orignal first fit the new secondfit
      j.firstfit = j.newnumber # make first fit the evolved new number
      results.append(j.newnumber) # add the number to the results list
      minnum = j.newnumber # make the last ball value the new minnum

  return results


In [ ]:
def loadballs(lotthist):
  """ 
  This mothod loads a list of ball objects with the lottery data passed in lotthist
  """
  lball=[]
  # load first 5 balls ball1..ball5
  for i in range(1,6):
    lball.append(Ball((f"ball{i}"),lotthist[(f"Ball{i}")]))
  # load the powerball
  lball.append(Ball(('powerball'),lotthist['powerball']))
  
  return lball

## Check the ticket numbers against the draw numbers

In [ ]:
def check_draw(histpd, checkpd):
  rows=0
  while rows !=checkpd.shape[0]:
    bmatch={}
    for j in checkpd.loc[:,'ball1':'powerball']:
      # print(f"{j} = {balls_check[j][rows]}")
      if j !='powerball' and (histpd.loc[:,'Ball1':'Ball5']==str(checkpd[j][rows])).all(0).any():
        bmatch[j]=str(checkpd[j][rows])
      elif j =='powerball' and (histpd.loc[:,'powerball']==str(checkpd[j][rows])).all(0).any():
        bmatch[j]=str(checkpd[j][rows])
      

    # print(bmatch)
    match_results=pd.DataFrame({'matched_balls':str(bmatch),'bfk':[checkpd['bpk'][rows]]})
    # print(f"{match_results}\n")
    match_results.to_sql('results',con=con,if_exists='append',index=False,index_label='rpk')
    rows+=1

## Main Program

In [ ]:
def main():
  # main code
  
  # open and read database
  con = sqlite3.connect(lotdbloc+dbname)
  # Load the data into a DataFrame
  names = pd.read_sql_query("SELECT * from names", con)
  balls = pd.read_sql_query("SELECT * from balls", con)
  ball_results = pd.read_sql_query("SELECT * from results", con)
  
    #get lotery history
  lotthist=get_numbers(powerballurl)
  # create a list of the six balls (of class 'Ball') loaded with the ball number frequecy data

  results=pd.DataFrame([[date.today()]+gagetvalues(loadballs(lotthist),generations)+[names.iloc[i][0]]for i in range(names.shape[0])])
  results.columns=['cdate','ball1','ball2','ball3','ball4','ball5','powerball','nfk']
  
  results.to_sql('balls',con=con,if_exists='append',index=False,index_label='bpk')
 
  for i in range(lotthist.shape[0]-2):
# for i in range(2):
    balls_check=balls[balls['cdate'].between(
      lotthist['Date'][i],
      lotthist['Date'][i+2]
      )]
    check_draw(lotthist['Date'][i+2],balls_check)
  
  con.close
  
    



In [ ]:
if __name__=='__main__' and config_check():
  main()

In [ ]:
lotthist=get_numbers(powerballurl)

lotthist

In [ ]:
[print(f"{x}: -- {gagetvalues(loadballs(lotthist),generations)}") for x in range(7)]



### 2nd Feb 23

0: -- [12, 30, 40, 53, 59, 13]

1: -- [25, 33, 35, 38, 56, 13]

2: -- [5, 7, 33, 53, 56, 18]

3: -- [15, 30, 37, 52, 57, 18]

4: -- [4, 15, 39, 47, 69, 11]

5: -- [8, 11, 36, 53, 59, 13]

6: -- [11, 33, 34, 55, 58, 18]


### sat 28th jan
0: -- [23, 43, 51, 56, 59, 18] *

1: -- [9, 17, 31, 53, 57, 13]

2: -- [23, 44, 45, 51, 59, 10] *

3: -- [6, 12, 35, 44, 59, 24]

4: -- [3, 30, 33, 39, 59, 10]

5: -- [1, 19, 41, 53, 58, 11]

6: -- [5, 15, 33, 52, 58, 18]


predicted results for thurs (26th jan 2023)

Winning numbers = 09	17	20	38	40	18

[2, 23, 35, 36, 62, 24]
[9, 28, 44, 53, 59, 11] *
[1, 12, 41, 45, 69, 10]
[4, 20, 38, 44, 58, 18] ** * = $7 = $1 Profit on 6 tickets
[8, 16, 40, 46, 69, 25] *
[35, 44, 45, 52, 58, 25]




## Database

### Tables

#### Names

In [ ]:
drop table names;
create table names
(
npk integer not null primary key AUTOINCREMENT,
fname varchar(50) not null,
lname varchar(50) not null,
phone varchar(50)
)
;

#### Balls

In [ ]:
drop table balls;
create table balls
(
bpk integer not null primary key AUTOINCREMENT,
cdate date not null,
ball1 integer not null,
ball2 integer not null,
ball3 integer not null,
ball4 integer not null,
ball5 integer not null,
powerball integer not null,
nfk integer not null,
foreign key (nfk) references names(npk)
)

#### Results

In [ ]:
drop table results;
Create table results
(
  rpk integer not null primary key AUTOINCREMENT,
  matched_balls varchar(100) not null,
  bfk integer not null,
  foreign key (bfk) references balls(bpk)
)